# Session 7: Complete Research Agent (45 minutes)

## 🎯 Learning Objectives
- Integrate all concepts from previous sessions
- Build a production-ready Research Assistant
- Implement RAG + Agents + Tools + Memory
- Best practices for deployment

## 📋 Final Goal
A Research Assistant that can:
- Answer questions from your documents (RAG)
- Use tools (search, calculator, etc.)
- Remember conversation context
- Make dynamic decisions

## ⏱️ Session Breakdown
- 10 min: Architecture overview
- 25 min: Building the complete agent
- 5 min: Testing & demo
- 5 min: Best practices & wrap-up

---

## 🆓 Using qwen2:0.5b

## 1. Setup - Bringing Everything Together

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "false"

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.runnables import RunnablePassthrough
from datetime import datetime
import json

# Initialize qwen2:0.5b via Ollama (LOCAL, FREE!)
llm = ChatOllama(
    model="qwen2:0.5b",
    temperature=0.7
)

print("✅ Session 7 Setup Complete!")
print("🖥️  Using qwen2:0.5b via Ollama - LOCAL, FREE, NO LIMITS!")

## 2. Initialize Models

In [ ]:
# Initialize embeddings model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ LLM: qwen2:0.5b (via Ollama)")
print("✅ Embeddings: sentence-transformers/all-MiniLM-L6-v2")

## 3. Architecture Overview

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                        COMPLETE RESEARCH AGENT                               │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│   USER INPUT                                                                 │
│       │                                                                      │
│       ▼                                                                      │
│   ┌───────────────────────────────────────────────────────────────────┐     │
│   │                         AGENT CORE                                 │     │
│   │   ┌─────────────────────────────────────────────────────────┐     │     │
│   │   │                   CONVERSATION MEMORY                    │     │     │
│   │   │   (Remembers context across interactions)                │     │     │
│   │   └─────────────────────────────────────────────────────────┘     │     │
│   │                            │                                       │     │
│   │                            ▼                                       │     │
│   │   ┌─────────────────────────────────────────────────────────┐     │     │
│   │   │                  DECISION ENGINE (LLM)                   │     │     │
│   │   │   - Understand user intent                               │     │     │
│   │   │   - Decide which tool(s) to use                          │     │     │
│   │   │   - Synthesize final response                            │     │     │
│   │   └─────────────────────────────────────────────────────────┘     │     │
│   │                            │                                       │     │
│   │              ┌─────────────┼─────────────┐                        │     │
│   │              │             │             │                        │     │
│   │              ▼             ▼             ▼                        │     │
│   │   ┌─────────────┐ ┌─────────────┐ ┌─────────────┐                │     │
│   │   │  RAG TOOL   │ │ CALCULATOR  │ │ SEARCH TOOL │                │     │
│   │   │ (Documents) │ │   (Math)    │ │   (Web)     │                │     │
│   │   └─────────────┘ └─────────────┘ └─────────────┘                │     │
│   └───────────────────────────────────────────────────────────────────┘     │
│                                                                              │
│       │                                                                      │
│       ▼                                                                      │
│   RESPONSE                                                                   │
│                                                                              │
└─────────────────────────────────────────────────────────────────────────────┘
```

## 4. Create Knowledge Base (RAG)

In [ ]:
# Sample knowledge base about AI/ML topics
knowledge_base = [
    Document(
        page_content="""LangChain is a framework for developing applications powered by language models.
        It provides tools for building chains, agents, and RAG systems. Key components include:
        - Prompts: Templates for LLM inputs
        - Models: Wrappers for various LLMs
        - Memory: Conversation context management
        - Agents: Dynamic decision-making systems""",
        metadata={"topic": "langchain", "type": "overview"}
    ),
    Document(
        page_content="""RAG (Retrieval Augmented Generation) combines retrieval and generation:
        1. Documents are split into chunks
        2. Chunks are embedded as vectors
        3. User query is embedded
        4. Similar chunks are retrieved
        5. Retrieved context + query → LLM → Answer
        This allows LLMs to answer questions about custom documents.""",
        metadata={"topic": "rag", "type": "explanation"}
    ),
    Document(
        page_content="""LangGraph is a library for building stateful, multi-step applications.
        Key concepts:
        - State: TypedDict that flows through the graph
        - Nodes: Functions that process state
        - Edges: Connections between nodes (can be conditional)
        - Checkpointing: Save/restore state for persistence""",
        metadata={"topic": "langgraph", "type": "overview"}
    ),
    Document(
        page_content="""Transformers are neural network architectures using self-attention.
        Key innovations:
        - Self-attention: Relates different positions in a sequence
        - Positional encoding: Adds position information
        - Multi-head attention: Attends to different aspects
        Popular models: BERT, GPT, T5, LLaMA, Mistral""",
        metadata={"topic": "transformers", "type": "explanation"}
    ),
    Document(
        page_content="""HuggingFace provides the largest repository of open-source ML models.
        Key products:
        - Hub: 500K+ models, datasets, spaces
        - Transformers: Library to load and run models
        - sentence-transformers: Embeddings library
        - Inference API: Cloud-hosted model inference""",
        metadata={"topic": "huggingface", "type": "overview"}
    )
]

# Create vector store
vectorstore = Chroma.from_documents(
    documents=knowledge_base,
    embedding=embeddings,
    collection_name="research_kb"
)

# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print(f"✅ Knowledge base created with {len(knowledge_base)} documents")

## 5. Create Tools

In [ ]:
@tool
def search_knowledge_base(query: str) -> str:
    """Search the knowledge base for information about AI/ML topics.
    Use this for questions about LangChain, RAG, LangGraph, Transformers, or HuggingFace.
    
    Args:
        query: The search query
    """
    docs = retriever.invoke(query)
    if docs:
        context = "\n\n".join([f"[{doc.metadata.get('topic', 'unknown')}]: {doc.page_content}" for doc in docs])
        return f"Found relevant information:\n\n{context}"
    return "No relevant information found in the knowledge base."

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression.
    Use this for any math calculations.
    
    Args:
        expression: A mathematical expression like '2 + 2' or '10 * 5 / 2'
    """
    try:
        allowed = set('0123456789+-*/(). ')
        if all(c in allowed for c in expression):
            result = eval(expression)
            return f"The result of {expression} is: {result}"
        return "Error: Invalid expression. Only numbers and basic operators (+, -, *, /) allowed."
    except Exception as e:
        return f"Error evaluating expression: {str(e)}"

@tool
def get_current_time() -> str:
    """Get the current date and time.
    Use this when the user asks about the current time or date.
    """
    now = datetime.now()
    return f"Current date and time: {now.strftime('%Y-%m-%d %H:%M:%S')}"

@tool
def web_search(query: str) -> str:
    """Search the web for general information.
    Use this for questions not covered by the knowledge base.
    
    Args:
        query: The search query
    """
    # Simulated web search
    return f"""Web search results for '{query}':
    [Simulated] Here is general information about {query}. 
    For accurate results, consider using a real search API like Tavily or SerpAPI."""

# Collect all tools
tools = [search_knowledge_base, calculator, get_current_time, web_search]

print("✅ Tools created:")
for t in tools:
    print(f"   🔧 {t.name}: {t.description[:60]}...")

## 6. Create the Agent

In [ ]:
# System prompt for the Research Assistant
system_prompt = """You are an AI Research Assistant specialized in helping users learn about AI/ML technologies.

Your capabilities:
1. Answer questions about LangChain, RAG, LangGraph, Transformers, and HuggingFace
2. Perform mathematical calculations
3. Tell the current time
4. Provide information

Guidelines:
- Be helpful, accurate, and concise
- If you don't know something, say so honestly
- Remember our conversation context
- Provide clear, educational responses suitable for learners"""

# Create a simple research agent without tool calling (Qwen 0.5B doesn't support it)
class SimpleResearchAgent:
    def __init__(self, llm, retriever, system_prompt: str):
        self.llm = llm
        self.retriever = retriever
        self.system_prompt = system_prompt
        self.memory = []
    
    def calculate(self, expr: str) -> str:
        """Simple calculator"""
        try:
            allowed = set('0123456789+-*/(). ')
            if all(c in allowed for c in expr):
                result = eval(expr)
                return f"The result of {expr} is: {result}"
            return "Invalid expression"
        except:
            return f"Error calculating {expr}"
    
    def get_time(self) -> str:
        """Get current time"""
        return f"Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
    
    def process(self, user_query: str) -> str:
        """Process user query with tool selection"""
        # Check what the user is asking about
        query_lower = user_query.lower()
        
        # Tool selection logic
        if any(word in query_lower for word in ['calculate', 'math', '+', '-', '*', '/', '=', 'multiply', 'divide']):
            # Try to extract math expression
            import re
            math_match = re.search(r'[\d\s+\-*/(.)]+', user_query)
            if math_match:
                return self.calculate(math_match.group())
        
        if any(word in query_lower for word in ['time', 'date', 'now', 'current']):
            return self.get_time()
        
        # Default: search knowledge base
        docs = self.retriever.invoke(user_query)
        context = "\n\n".join([
            f"[{doc.metadata.get('topic', 'unknown')}]: {doc.page_content}" 
            for doc in docs
        ]) if docs else "No information found."
        
        # Use RAG context to answer
        prompt = ChatPromptTemplate.from_template(
            self.system_prompt + "\n\nContext from knowledge base:\n{context}\n\nUser question: {query}\n\nAnswer:"
        )
        chain = prompt | self.llm | StrOutputParser()
        
        response = chain.invoke({
            "context": context,
            "query": user_query
        })
        
        # Store in memory
        self.memory.append({"user": user_query, "assistant": response})
        
        return response

# Create agent
research_agent = SimpleResearchAgent(llm, retriever, system_prompt)

print("✅ Research Agent created with:")
print("   📚 Knowledge base integration")
print("   🔧 Math & time tools")
print("   🧠 Conversation memory")

## 7. Create the Chat Interface

In [ ]:
class ResearchAssistant:
    """Complete Research Assistant with RAG, tools, and memory."""
    
    def __init__(self, agent):
        self.agent = agent
    
    def chat(self, message: str) -> str:
        """Send a message and get a response."""
        return self.agent.process(message)
    
    def get_memory(self) -> list:
        """Get conversation memory"""
        return self.agent.memory

# Create assistant instance
assistant = ResearchAssistant(research_agent)

print("✅ Research Assistant ready!")

## 8. Demo: Test the Complete Agent

In [ ]:
print("\n" + "="*60)
print("🤖 RESEARCH ASSISTANT DEMO")
print("="*60)

# Test 1: Knowledge base query
print("\n📝 Test 1: Knowledge Base Query")
print("-" * 40)
response = assistant.chat("What is RAG and how does it work?")
print(f"User: What is RAG and how does it work?")
print(f"Assistant: {response}")

In [ ]:
# Test 2: Calculator
print("\n📝 Test 2: Calculator")
print("-" * 40)
response = assistant.chat("Calculate 15 * 24 + 100")
print(f"User: Calculate 15 * 24 + 100")
print(f"Assistant: {response}")

In [ ]:
# Test 3: Current time
print("\n📝 Test 3: Current Time")
print("-" * 40)
response = assistant.chat("What time is it now?")
print(f"User: What time is it now?")
print(f"Assistant: {response}")

In [ ]:
# Test 4: Multi-step query
print("\n📝 Test 4: Complex Query")
print("-" * 40)
response = assistant.chat("Compare LangChain and LangGraph - what are the main differences?")
print(f"User: Compare LangChain and LangGraph - what are the main differences?")
print(f"Assistant: {response}")

In [ ]:
# Test 5: Memory test
print("\n📝 Test 5: Memory Test")
print("-" * 40)
response = assistant.chat("Based on our conversation, what topics have we discussed?")
print(f"User: Based on our conversation, what topics have we discussed?")
print(f"Assistant: {response}")

## 9. Interactive Chat Loop

In [ ]:
# Interactive chat (uncomment to use)

print("\n" + "="*60)
print("🤖 INTERACTIVE RESEARCH ASSISTANT")
print("="*60)
print("Type 'quit' to exit\n")

while True:
    user_input = input("You: ")
    if user_input.lower() in ['quit', 'exit', 'q']:
        print("\nGoodbye! 👋")
        break
    
    response = assistant.chat(user_input)
    print(f"Assistant: {response}\n")


## 10. Production Best Practices

In [ ]:
print("""
🏭 PRODUCTION BEST PRACTICES
════════════════════════════════════════════════════════════════

1. ERROR HANDLING
   ─────────────
   - Wrap all tool calls in try/except
   - Implement retry logic with exponential backoff
   - Have fallback responses for failures

2. MONITORING (LangSmith)
   ─────────────────────
   - Enable tracing for all production requests
   - Tag traces by environment, user, version
   - Set up alerts for error rates and latency

3. SECURITY
   ────────
   - Never expose API keys in code
   - Use environment variables or secret managers
   - Sanitize user inputs before tool calls
   - Implement rate limiting

4. PERFORMANCE
   ───────────
   - Cache frequent queries
   - Use streaming for long responses
   - Consider async operations for tools
   - Optimize chunk sizes for RAG

5. SCALABILITY
   ───────────
   - Use persistent vector stores (Pinecone, Weaviate)
   - Implement database-backed memory
   - Consider serverless deployment
   - Load balance across LLM providers

6. EVALUATION
   ──────────
   - Create golden test datasets
   - Run automated evaluations before deploy
   - Track quality metrics over time
   - A/B test prompt changes

════════════════════════════════════════════════════════════════
""")

## 📚 Workshop Complete!

### What You've Learned:

| Session | Topic | Key Concepts |
|---------|-------|-------------|
| 1 | LangChain Basics | LLMs, Prompts, LCEL, Output Parsers |
| 2 | Chains & Memory | Sequential chains, Conversation memory |
| 3 | RAG | Document loading, Embeddings, Vector stores |
| 4 | LangSmith | Tracing, Debugging, Evaluation |
| 5 | LangGraph | Agents, Tools, State machines |
| 6 | HuggingFace | Transformers, Local models, Pipelines |
| 7 | Complete Agent | Integration, Production best practices |

### Next Steps:

1. **Practice**: Build your own agent with custom tools
2. **Extend**: Add more knowledge to the RAG system
3. **Deploy**: Try deploying with LangServe or FastAPI
4. **Evaluate**: Set up LangSmith for your projects
5. **Explore**: Try different LLM providers and models

### Resources:

- [LangChain Docs](https://python.langchain.com/docs/)
- [LangGraph Docs](https://langchain-ai.github.io/langgraph/)
- [LangSmith](https://smith.langchain.com/)
- [HuggingFace Hub](https://huggingface.co/)
- [Google AI Studio](https://aistudio.google.com/) (Free Gemini API)

In [ ]:
print("""
╔═══════════════════════════════════════════════════════════════════════════╗
║                                                                            ║
║     🎉 CONGRATULATIONS! WORKSHOP COMPLETE! 🎉                               ║
║                                                                            ║
╠════════════════════════════════════════════════════════════════════════════╣
║                                                                            ║
║     You've built a complete Research Assistant with:                        ║
║                                                                            ║
║     ✅ LangChain for LLM orchestration                                      ║
║     ✅ RAG for document Q&A                                                 ║
║     ✅ LangGraph for agent workflows                                        ║
║     ✅ Multiple specialized tools                                           ║
║     ✅ Conversation memory                                                  ║
║     ✅ HuggingFace embeddings                                               ║
║     ✅ Free Gemma 3 27B via Gemini API                                      ║
║                                                                            ║
╠════════════════════════════════════════════════════════════════════════════╣
║                                                                            ║
║     Thank you for attending! Happy building! 🚀                             ║
║                                                                            ║
╚═══════════════════════════════════════════════════════════════════════════╝
""")